# Run Classification Pipeline

This notebook runs a configurable classification pipeline on standardized head-metrics datasets. Model selection happens only on the train file; the test file is evaluated once after selecting the best configuration.

## 1. Configure Experiment

Edit the JSON config path or modify the config object directly in the next cell. The default config points to the bundled SUS demo files in `notebook/data/generated`. For binary tasks, targets must be encoded as `0/1`.

In [ ]:
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    NOTEBOOK_DIR = Path('notebook')

sys.path.insert(0, str((NOTEBOOK_DIR / 'scripts').resolve()))

CONFIG_PATH = NOTEBOOK_DIR / 'configs' / 'classification_config_fast_sus_demo.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

# Common edits. Keep these paths relative to NOTEBOOK_DIR so the notebook remains GitHub-portable.
config['train_path'] = str(NOTEBOOK_DIR / 'data' / 'generated' / 'FAST_SUS_BuildA_standardized.xlsx')
config['test_path'] = str(NOTEBOOK_DIR / 'data' / 'generated' / 'FAST_SUS_BuildB_standardized.xlsx')
config['target_column'] = 'sus_not_acceptable_target'
config['classification_type'] = 'binary'
config['selection_metric'] = 'f1_positive'
config['output_dir'] = str(NOTEBOOK_DIR / 'outputs' / 'fast_sus_demo_pipeline')

# Optional: restrict to feature families. Use None for all 47 features.
config['feature_families'] = None

# Optional: make a faster first run.
# config['models'] = [{'name': 'logreg', 'param_grid': [{'C': 1.0}]}]
# config['imbalance_strategies'] = ['none', 'class_weight', 'random_undersample']
# config['threshold_strategies'] = ['default', 'roc_gmean']

RUN_CONFIG_PATH = NOTEBOOK_DIR / 'outputs' / 'classification_config_run.json'
RUN_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(RUN_CONFIG_PATH)

## 2. Review Candidate Search Space

This cell shows which models, preprocessing strategies, imbalance strategies, and threshold strategies will be tested.

In [ ]:
print('Models:')
for model in config['models']:
    print('-', model['name'], 'configs:', len(model.get('param_grid', [{}])))

print('\nPreprocessing:', config['preprocessing'])
print('Imbalance strategies:', config['imbalance_strategies'])
print('Threshold strategies:', config['threshold_strategies'])
print('CV:', config['cv_splits'], 'folds x', config['cv_repeats'], 'repeats')

## 3. Run Pipeline

The script performs repeated stratified CV on the train dataset. Resampling is applied only inside training folds. Thresholds are estimated from training-fold scores and transferred to the test dataset.

In [ ]:
import subprocess

cmd = [sys.executable, str(NOTEBOOK_DIR / 'scripts' / 'classification_pipeline.py'), '--config', str(RUN_CONFIG_PATH)]
print(' '.join(cmd))
completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
print(completed.stdout[-4000:])

## 4. Inspect Results

The main outputs are `cv_results.csv`, `test_metrics.json`, `test_predictions.csv`, and `best_model.pkl`.

In [ ]:
import pandas as pd

output_dir = Path(config['output_dir'])
cv_results = pd.read_csv(output_dir / 'cv_results.csv')
metrics = json.loads((output_dir / 'test_metrics.json').read_text(encoding='utf-8'))

display(cv_results.head(10))
print(json.dumps(metrics['test_metrics'], indent=2))
print('\nBest candidate:')
print(json.dumps(metrics['best_candidate'], indent=2))

## 5. Confusion Matrix From Test Predictions

Use this quick diagnostic to understand the selected model's errors on the test dataset.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

predictions = pd.read_csv(output_dir / 'test_predictions.csv')
cm = confusion_matrix(predictions['y_true'], predictions['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')
plt.title('Test Confusion Matrix')
plt.show()